In [1]:
import os
import json
import sys
import subprocess
from pathlib import Path
from collections import defaultdict

# --- CONFIGURATION ---
ROOT_DIR = "."
SIMAL_CLI_PATH = "simal_cli.py"  # Path to SiMAL src folder
JSON_EXTENSIONS = {".txt", ".json"}
SIMAL_EXTENSIONS = {".txt", ".simal"} 

def get_validation_mode(folder_name):
    """
    Determines if a folder contains 'json' or 'simal' files based on the name.
    """
    lower_name = folder_name.lower()
    if "simal" in lower_name:
        return "simal"
    return "json"

def validate_json_file(file_path):
    """Returns None if valid, else returns error string."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            json.load(f)
        return None
    except json.JSONDecodeError as e:
        return f"JSON Syntax Error: {e}"
    except Exception as e:
        return f"System Error: {e}"

def validate_simal_file(file_path):
    """Returns None if valid, else returns error string from CLI output."""
    try:
        # Run the CLI script with a timeout
        result = subprocess.run(
            [sys.executable, SIMAL_CLI_PATH, str(file_path)],
            capture_output=True,
            text=True,
            timeout=10
        )
        
        if result.returncode == 0:
            return None
        else:
            return result.stdout.strip() or result.stderr.strip() or "Unknown CLI failure"

    except subprocess.TimeoutExpired:
        # Handle the hang gracefully
        return f"TIMEOUT: Execution exceeded {10}s"
            
    except Exception as e:
        return f"Subprocess Call Error: {e}"

def validate_directory(root_path):
    report = defaultdict(lambda: {
        "mode": "", 
        "total": 0, 
        "passed": 0, 
        "failed": 0, 
        "failures": []
    })

    # Verify CLI script exists if we plan to use it
    if not os.path.exists(SIMAL_CLI_PATH):
        print(f"WARNING: {SIMAL_CLI_PATH} not found. SiMAL validation will fail.")

    for dirpath, _, filenames in os.walk(root_path):
        folder_name = os.path.basename(dirpath)
        
        if folder_name.startswith(".") or folder_name == "__pycache__" or folder_name == "repo":
            continue

        # filter out dirs that have 'repo' dir anywhere in their path
        if "repo" in dirpath.split(os.sep):
            continue

        print("Scanning folder:", dirpath)

        # Determine mode for this folder
        mode = get_validation_mode(folder_name)
        
        # Select target extensions based on mode
        target_exts = SIMAL_EXTENSIONS if mode == "simal" else JSON_EXTENSIONS

        for filename in filenames:
            file_path = Path(dirpath) / filename
            
            if file_path.suffix not in target_exts:
                continue
            
            # Skip the scripts themselves if they are in the scan path
            if filename in ["validate_mixed.py", SIMAL_CLI_PATH]:
                continue

            error = None
            if mode == "json":
                error = validate_json_file(file_path)
            elif mode == "simal":
                error = validate_simal_file(file_path)

            # Update stats
            stats = report[dirpath]
            stats["mode"] = mode
            stats["total"] += 1
            
            if error:
                stats["failed"] += 1
                stats["failures"].append({"file": filename, "error": error})
            else:
                stats["passed"] += 1

    return report

def print_summary(report):
    print(f"\n{'='*70}")
    print(f"{'VALIDATION REPORT':^70}")
    print(f"{'='*70}")

    global_json_fail = 0
    global_simal_fail = 0

    if not report:
        print("No matching files found.")
        return

    # Sort by mode then path for cleaner output
    sorted_paths = sorted(report.keys(), key=lambda p: (report[p]['mode'], p))

    for path in sorted_paths:
        data = report[path]
        if data['total'] == 0: continue

        mode_icon = "🔷" if data['mode'] == "json" else "🔶"
        status_icon = "❌" if data['failed'] > 0 else "✅"
        
        print(f"\n{mode_icon} [{data['mode'].upper()}] {os.path.basename(path)}")
        print(f"   Path: {path}")
        print(f"   Stats: {data['total']} files | {status_icon} {data['failed']} failed")

        if data['failed'] > 0:
            print("   Failures:")
            for fail in data['failures']:
                # Truncate very long error messages from CLI
                err_msg = fail['error'].replace('\n', ' ')
                if len(err_msg) > 200: err_msg = err_msg[:197] + "..."
                print(f"      • {fail['file']} -> {err_msg}")
            
            if data['mode'] == "json": global_json_fail += data['failed']
            else: global_simal_fail += data['failed']

    print(f"\n{'='*70}")
    print(f"SUMMARY")
    print(f"   JSON Failures:  {global_json_fail}")
    print(f"   SiMAL Failures: {global_simal_fail}")
    print(f"{'='*70}")

In [ ]:
results = validate_directory("./simal_schemas")
print_summary(results)